# Lab 6 - SQL with DuckDB

<a href="https://colab.research.google.com/github/gaulinmp/AccountingDataAnalytics/blob/main/labs_hw/week6_ETL-SQL-LLM/Lab-6_SQL-with-DuckDB.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook will walk you through loading two datasets and joining them together with SQL. We will cover the following steps:

1. Loading the Compustat (annual financial statements) and CRSP (daily stock prices) data
2. Querying the data with SQL, using a library called DuckDB

The intent of the notebook is for you to execute each cell in order. Before you start, drag both `lab6_data_compustat.csv` and `lab6_data_crsp.csv` into your Colab page (see Lab 1 for instructions).

In [1]:
# The usual imports: pathlib to point to files, and pandas to load them
from pathlib import Path
import pandas as pd
# This one is new. DuckDB is a small, fast database that lives inside Python.
# It lets us write SQL queries directly against pandas DataFrames.
import duckdb

In [3]:
from google.colab import files

uploaded = files.upload()

Saving lab6_data_compustat.csv to lab6_data_compustat (1).csv
Saving lab6_data_crsp.csv to lab6_data_crsp.csv


In [4]:
# Point to the files, and make sure they exist
compustat_path = Path("lab6_data_compustat.csv")
crsp_path = Path("lab6_data_crsp.csv")
compustat_path.exists(), crsp_path.exists()

(True, True)

That should say `(True, True)` above. If either says `False`, then that file is not loaded correctly. Please make sure to drag both CSV files into your Colab page.

Okay, now to actually load the data. Notice the `parse_dates` argument: it tells pandas which columns are dates, so they get loaded as dates instead of text. That matters later, because we're going to compare dates in our SQL query.

In [5]:
# Load Compustat: one row per company per fiscal year
compustat = pd.read_csv(compustat_path, parse_dates=["fiscal_year_end", "fiscal_year_end_prev", "fiscal_year_end_next",
                                                     "earn_annc_date", "earn_annc_date_prev", "earn_annc_date_next"])
compustat.head()

,name,fyear,ticker,permno,fyr,earn_annc_date,age_days,fiscal_year_end,at,act,...,bve,bign,auditor,auop,fiscal_year_end_prev,earn_annc_date_prev,fiscal_year_end_next,earn_annc_date_next,gics_sector,gics_sector_name
0,AMERICAN AIRLINES GROUP INC,2000,AMR,21020,12,2001-01-17,18263,2000-12-31,26213.0,5179.0,...,7176.0,1,Ernst & Young,Unqualified,1999-12-31,2000-01-19,2001-12-31,2002-01-16,20.0,Industrials
1,AMERICAN AIRLINES GROUP INC,2001,AMR,21020,12,2002-01-16,18628,2001-12-31,32841.0,6540.0,...,5373.0,1,Ernst & Young,Unqualified with additional language,2000-12-31,2001-01-17,2002-12-31,2003-01-22,20.0,Industrials
2,AMERICAN AIRLINES GROUP INC,2002,AMR,21020,12,2003-01-22,18993,2002-12-31,30267.0,4937.0,...,957.0,1,Ernst & Young,Unqualified with additional language,2001-12-31,2002-01-16,2003-12-31,2004-01-21,20.0,Industrials
3,AMERICAN AIRLINES GROUP INC,2003,AMR,21020,12,2004-01-21,19358,2003-12-31,29330.0,4682.0,...,46.0,1,Ernst & Young,Unqualified with additional language,2002-12-31,2003-01-22,2004-12-31,2005-01-19,20.0,Industrials
4,AMERICAN AIRLINES GROUP INC,2004,AMR,21020,12,2005-01-19,19724,2004-12-31,28773.0,4971.0,...,-581.0,1,Ernst & Young,Unqualified with additional language,2003-12-31,2004-01-21,2005-12-31,2006-01-18,20.0,Industrials


In [6]:
# Load CRSP: one row per company per trading day
crsp = pd.read_csv(crsp_path, parse_dates=["date"])
crsp.head()

,permno,ticker,date,ret,prc,shrout,vol,bid,ask,shrcls
0,10107,MSFT,1999-01-04,0.016674,141.000,2470000,17868249,140.875,140.9375,A
1,10107,MSFT,1999-01-05,0.039007,146.500,2470000,16424412,146.375,146.4375,A
2,10107,MSFT,1999-01-06,0.032423,151.250,2470000,18025141,151.375,151.4375,A
3,10107,MSFT,1999-01-07,-0.004959,150.500,2470000,14179585,150.500,150.5000,A
4,10107,MSFT,1999-01-08,-0.004153,149.875,2470000,12709729,149.875,149.9375,A


Take a second to notice how different these two tables are. Compustat has one row per firm-*year* (a few hundred rows), while CRSP has one row per firm-*day* (over 150,000 rows). Let's confirm:

In [7]:
print("Compustat:", compustat.shape)
print("CRSP:     ", crsp.shape)

Compustat: (594, 49)
CRSP:      (156002, 10)


Both tables have a `permno` column, which is CRSP's permanent identifier for a stock. That's our key for linking the two tables together. (Why not `ticker`? Because tickers change. American Airlines was `AMR` before it was `AAL`, but its `permno` stayed the same.)

## SQL with DuckDB

Now for the fun part. We want to match each firm's *daily* stock data to the *annual* financial statements that were current on that day. In other words, for each fiscal year, grab every trading day from that fiscal year end up to (but not including) the next one.

That's a join on `permno`, plus a date range condition. Here's the SQL:

```sql
-- Example Query: Join Compustat and CRSP on Permno and Date between fiscal year end dates
SELECT *
FROM compustat
JOIN crsp
    ON compustat.permno = crsp.permno
        AND date >= fiscal_year_end
        AND date < fiscal_year_end_next
LIMIT 5
```

The neat trick with DuckDB is that the table names in the query (`compustat` and `crsp`) are just the names of our pandas DataFrames. DuckDB finds them automatically. We write the query as a Python string, hand it to `duckdb.sql()`, and convert the result back to a DataFrame with `.df()`.

In [8]:
query = """
-- Example Query: Join Compustat and CRSP on Permno and Date between fiscal year end dates
SELECT *
FROM compustat
JOIN crsp
    ON compustat.permno = crsp.permno
        AND date >= fiscal_year_end
        AND date < fiscal_year_end_next
LIMIT 5
"""

duckdb.sql(query).df()

,name,fyear,ticker,permno,fyr,earn_annc_date,age_days,fiscal_year_end,at,act,...,permno_1,ticker_1,date,ret,prc,shrout,vol,bid,ask,shrcls
0,MICROSOFT CORP,2006,MSFT,10107,6,2006-07-20,7670,2006-06-30,69597.0,49010.0,...,10107,MSFT,2006-06-30,-0.007252,23.30,10062000,73495446,23.38,23.31,A
1,MICROSOFT CORP,2006,MSFT,10107,6,2006-07-20,7670,2006-06-30,69597.0,49010.0,...,10107,MSFT,2006-07-03,0.017167,23.70,10062000,25744996,23.67,23.68,A
2,MICROSOFT CORP,2006,MSFT,10107,6,2006-07-20,7670,2006-06-30,69597.0,49010.0,...,10107,MSFT,2006-07-05,-0.014768,23.35,10062000,53172363,23.33,23.31,A
3,MICROSOFT CORP,2006,MSFT,10107,6,2006-07-20,7670,2006-06-30,69597.0,49010.0,...,10107,MSFT,2006-07-06,0.005567,23.48,10062000,63802278,23.46,23.47,A
4,MICROSOFT CORP,2006,MSFT,10107,6,2006-07-20,7670,2006-06-30,69597.0,49010.0,...,10107,MSFT,2006-07-07,-0.007666,23.30,10062000,65168568,23.31,23.31,A


Scroll all the way to the right in that output. You'll see columns named `permno_1` and `ticker_1`. That's because both tables have a `permno` and `ticker` column, and `SELECT *` grabs everything from both. DuckDB renames the duplicates so they don't collide.

In practice, you'd usually pick just the columns you want instead of `*`. Here's the same join, returning only the firm name, fiscal year, net income (`ni`), the trading date, and the daily return (`ret`). Notice the `table.column` syntax, which tells SQL which table each column comes from.

In [9]:
query = """
SELECT compustat.name, compustat.fyear, compustat.ni, crsp.date, crsp.ret
FROM compustat
JOIN crsp
    ON compustat.permno = crsp.permno
        AND date >= fiscal_year_end
        AND date < fiscal_year_end_next
LIMIT 5
"""

duckdb.sql(query).df()

,name,fyear,ni,date,ret
0,MICROSOFT CORP,2006,12599.0,2006-06-30,-0.007252
1,MICROSOFT CORP,2006,12599.0,2006-07-03,0.017167
2,MICROSOFT CORP,2006,12599.0,2006-07-05,-0.014768
3,MICROSOFT CORP,2006,12599.0,2006-07-06,0.005567
4,MICROSOFT CORP,2006,12599.0,2006-07-07,-0.007666


Now you're writing SQL. From here, you can add a `WHERE` clause to filter, `GROUP BY` to aggregate (e.g., the average daily return per firm-year), or `ORDER BY` to sort. And remember: if you get stuck, write a comment describing what you want, and let Gemini suggest the query.

In [10]:
query = """
SELECT *
FROM compustat
JOIN crsp
    ON compustat.permno = crsp.permno
    AND crsp.date = compustat.fiscal_year_end
"""

one_day_merge = duckdb.sql(query).df()

one_day_merge.head()

,name,fyear,ticker,permno,fyr,earn_annc_date,age_days,fiscal_year_end,at,act,...,permno_1,ticker_1,date,ret,prc,shrout,vol,bid,ask,shrcls
0,MICROSOFT CORP,2000,MSFT,10107,6,2000-07-18,5479,2000-06-30,52150.0,30308.0,...,10107,MSFT,2000-06-30,0.036437,80.00,5262405,26862582,79.9375,80.00,A
1,MICROSOFT CORP,2003,MSFT,10107,6,2003-07-17,6574,2003-06-30,79571.0,58973.0,...,10107,MSFT,2003-06-30,0.000390,25.64,10718000,54371120,25.6100,25.61,A
2,MICROSOFT CORP,2004,MSFT,10107,6,2004-07-22,6940,2004-06-30,92389.0,70566.0,...,10107,MSFT,2004-06-30,0.002105,28.56,10862000,84035310,28.5900,28.57,A
3,MICROSOFT CORP,2005,MSFT,10107,6,2005-07-21,7305,2005-06-30,70815.0,48737.0,...,10107,MSFT,2005-06-30,-0.009964,24.84,10710000,82195698,24.8300,24.83,A
4,MICROSOFT CORP,2006,MSFT,10107,6,2006-07-20,7670,2006-06-30,69597.0,49010.0,...,10107,MSFT,2006-06-30,-0.007252,23.30,10062000,73495446,23.3800,23.31,A


In [11]:
import duckdb

query = """
SELECT
    c.*,

    -- Day -1: last trading day before the earnings announcement
    m1.date AS date_minus_1,
    m1.ret  AS return_minus_1,

    -- Day 0: EA date, or first trading day after if EA is not a trading day
    d0.date AS date_0,
    d0.ret  AS return_0,

    -- Day +1: first trading day after Day 0
    p1.date AS date_plus_1,
    p1.ret  AS return_plus_1

FROM compustat AS c

LEFT JOIN LATERAL (
    SELECT
        r.date,
        r.ret
    FROM crsp AS r
    WHERE r.permno = c.permno
      AND r.date < c.earn_annc_date
    ORDER BY r.date DESC
    LIMIT 1
) AS m1
ON TRUE

LEFT JOIN LATERAL (
    SELECT
        r.date,
        r.ret
    FROM crsp AS r
    WHERE r.permno = c.permno
      AND r.date >= c.earn_annc_date
    ORDER BY r.date ASC
    LIMIT 1
) AS d0
ON TRUE

LEFT JOIN LATERAL (
    SELECT
        r.date,
        r.ret
    FROM crsp AS r
    WHERE r.permno = c.permno
      AND r.date > d0.date
    ORDER BY r.date ASC
    LIMIT 1
) AS p1
ON TRUE
"""

ea_returns = duckdb.sql(query).df()

ea_returns.head()

,name,fyear,ticker,permno,fyr,earn_annc_date,age_days,fiscal_year_end,at,act,...,fiscal_year_end_next,earn_annc_date_next,gics_sector,gics_sector_name,date_minus_1,return_minus_1,date_0,return_0,date_plus_1,return_plus_1
0,MICROSOFT CORP,2001,MSFT,10107,6,2001-07-19,5844,2001-06-30,59257.0,39637.0,...,2002-06-30,2002-07-18,45.0,Information Technology,2001-07-18,-0.017405,2001-07-19,0.028341,2001-07-20,-0.046714
1,MICROSOFT CORP,2016,MSFT,10107,6,2016-07-19,11323,2016-06-30,193694.0,139660.0,...,2017-06-30,2017-07-20,45.0,Information Technology,2016-07-18,0.004842,2016-07-19,-0.016123,2016-07-20,0.053117
2,COCA-COLA CO,2009,KO,11308,12,2010-02-09,21550,2009-12-31,48671.0,17551.0,...,2010-12-31,2011-02-09,30.0,Consumer Staples,2010-02-08,-0.008288,2010-02-09,0.025831,2010-02-10,-0.004073
3,EXXONMOBIL HOLDINGS CORP,2000,XOM,11850,12,2001-01-24,18263,2000-12-31,149000.0,40399.0,...,2001-12-31,2002-01-23,10.0,Energy,2001-01-23,0.019470,2001-01-24,-0.002292,2001-01-25,0.010720
4,EXXONMOBIL HOLDINGS CORP,2012,XOM,11850,12,2013-02-01,22646,2012-12-31,333795.0,64460.0,...,2013-12-31,2014-01-30,10.0,Energy,2013-01-31,-0.007720,2013-02-01,0.000778,2013-02-04,-0.009885


In [12]:
ea_returns[
    [
        "permno",
        "earn_annc_date",
        "date_minus_1",
        "return_minus_1",
        "date_0",
        "return_0",
        "date_plus_1",
        "return_plus_1"
    ]
].head(10)

,permno,earn_annc_date,date_minus_1,return_minus_1,date_0,return_0,date_plus_1,return_plus_1
0,10107,2001-07-19,2001-07-18,-0.017405,2001-07-19,0.028341,2001-07-20,-0.046714
1,10107,2016-07-19,2016-07-18,0.004842,2016-07-19,-0.016123,2016-07-20,0.053117
2,11308,2010-02-09,2010-02-08,-0.008288,2010-02-09,0.025831,2010-02-10,-0.004073
3,11850,2001-01-24,2001-01-23,0.019470,2001-01-24,-0.002292,2001-01-25,0.010720
4,11850,2013-02-01,2013-01-31,-0.007720,2013-02-01,0.000778,2013-02-04,-0.009885
5,12079,2007-03-14,2007-03-13,-0.025862,2007-03-14,-0.008522,2007-03-15,-0.028760
6,12369,2022-02-01,2022-01-31,0.049562,2022-02-01,0.025413,2022-02-02,-0.010542
7,14541,2003-01-31,2003-01-30,-0.016845,2003-01-31,0.003115,2003-02-03,0.019410
8,14541,2014-01-31,2014-01-30,0.003879,2014-01-31,-0.041391,2014-02-03,-0.004390
9,14541,2017-01-27,2017-01-26,-0.005885,2017-01-27,-0.023681,2017-01-30,-0.017313


In [14]:
query = """
SELECT
    c.*,
    annual.cumulative_return
FROM compustat AS c

LEFT JOIN LATERAL (
    SELECT
        PRODUCT(1 + r.ret) - 1 AS cumulative_return
    FROM crsp AS r
    WHERE r.permno = c.permno
      AND r.date > c.earn_annc_date_prev
      AND r.date <= c.earn_annc_date
) AS annual
ON TRUE
"""

annual_returns = duckdb.sql(query).df()

annual_returns.head()

,name,fyear,ticker,permno,fyr,earn_annc_date,age_days,fiscal_year_end,at,act,...,bign,auditor,auop,fiscal_year_end_prev,earn_annc_date_prev,fiscal_year_end_next,earn_annc_date_next,gics_sector,gics_sector_name,cumulative_return
0,BANK OF AMERICA CORP,2011,BAC,59408,12,2012-01-19,19358,2011-12-31,2129046.0,NaN,...,1,PricewaterhouseCoopers,Unqualified,2010-12-31,2011-01-21,2012-12-31,2013-01-17,40.0,Financials,-0.509292
1,BANK OF AMERICA CORP,2010,BAC,59408,12,2011-01-21,18993,2010-12-31,2264909.0,NaN,...,1,PricewaterhouseCoopers,Unqualified,2009-12-31,2010-01-20,2011-12-31,2012-01-19,40.0,Financials,-0.133349
2,BANK OF AMERICA CORP,2009,BAC,59408,12,2010-01-20,18628,2009-12-31,2223299.0,NaN,...,1,PricewaterhouseCoopers,Unqualified,2008-12-31,2009-01-16,2010-12-31,2011-01-21,40.0,Financials,1.307984
3,BANK OF AMERICA CORP,2008,BAC,59408,12,2009-01-16,18263,2008-12-31,1817943.0,NaN,...,1,PricewaterhouseCoopers,Unqualified,2007-12-31,2008-01-22,2009-12-31,2010-01-20,40.0,Financials,-0.792603
4,BANK OF AMERICA CORP,2007,BAC,59408,12,2008-01-22,17897,2007-12-31,1715746.0,NaN,...,1,PricewaterhouseCoopers,Unqualified with additional language,2006-12-31,2007-01-23,2008-12-31,2009-01-16,40.0,Financials,-0.263760


In [13]:
query = """
SELECT
    c.*,
    annual.trading_days,
    annual.cumulative_return
FROM compustat AS c

LEFT JOIN LATERAL (
    SELECT
        COUNT(r.ret) AS trading_days,
        PRODUCT(1 + r.ret) - 1 AS cumulative_return
    FROM crsp AS r
    WHERE r.permno = c.permno
      AND r.date > c.earn_annc_date_prev
      AND r.date <= c.earn_annc_date
) AS annual
ON TRUE
"""

annual_returns = duckdb.sql(query).df()

annual_returns[
    [
        "permno",
        "earn_annc_date_prev",
        "earn_annc_date",
        "trading_days",
        "cumulative_return"
    ]
].head(10)

,permno,earn_annc_date_prev,earn_annc_date,trading_days,cumulative_return
0,59408,2011-01-21,2012-01-19,250,-0.509292
1,59408,2010-01-20,2011-01-21,254,-0.133349
2,59408,2009-01-16,2010-01-20,253,1.307984
3,59408,2008-01-22,2009-01-16,250,-0.792603
4,59408,2007-01-23,2008-01-22,251,-0.263760
5,59408,2006-01-23,2007-01-23,251,0.265287
6,59408,2019-01-16,2020-01-15,251,0.246124
7,59408,2017-01-13,2018-01-17,253,0.376286
8,59408,2016-01-19,2017-01-13,250,0.640726
9,59408,2015-01-15,2016-01-19,253,-0.051766
